In [11]:
import torch
import gc
import wandb
import warnings
import optuna

import torch.nn.functional as F
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import ndcg_score
from torch.optim.lr_scheduler import StepLR

In [4]:
class TextRankingDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=512):
        print("Pre-tokenizing dataset (this takes a minute)...")
        
        cv_texts = dataframe['cv_text'].astype(str).tolist()
        vac_texts = dataframe['vacancy_text'].astype(str).tolist()
        
        # Tokenize everything upfront
        self.cv_encodings = tokenizer(
            cv_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )
        self.vac_encodings = tokenizer(
            vac_texts, add_special_tokens=True, 
            max_length=max_length, padding='max_length', 
            truncation=True, return_tensors='pt'
        )

        self.labels = dataframe['response'].values
        self.cvids = dataframe['cvid'].astype(str).tolist()
        self.vacancy_ids = dataframe['humanjobid'].astype(str).tolist()
        
        # --- THE CORE DIFFERENCE: GROUP BY CANDIDATE ---
        self.unique_cvids = dataframe['cvid'].unique().tolist()
        self.grouped_indices = dataframe.groupby('cvid').indices

    def __len__(self):
        # Length is now the number of unique candidates, NOT the number of total rows
        return len(self.unique_cvids)

    def __getitem__(self, idx):
        # 1. Look up the candidate
        cvid = self.unique_cvids[idx]
        # 2. Get the specific row indices for all N of their vacancies
        indices = self.grouped_indices[cvid] 
        
        # 3. Return the N-sized chunk of tensors for this single candidate
        return {
            'cv_input_ids': self.cv_encodings['input_ids'][indices],
            'cv_attention_mask': self.cv_encodings['attention_mask'][indices],
            'vac_input_ids': self.vac_encodings['input_ids'][indices],
            'vac_attention_mask': self.vac_encodings['attention_mask'][indices],
            'labels': torch.tensor(self.labels[indices], dtype=torch.float32),
            'cvid': [self.cvids[i] for i in indices],
            'vacancy_id': [self.vacancy_ids[i] for i in indices]
        }

def ranking_collate_fn(batch):
    """
    Takes a list of candidate dictionaries (where each dict contains N items)
    and concatenates them along the 0th dimension to mimic PyG batching.
    """
    return {
        'cv_input_ids': torch.cat([b['cv_input_ids'] for b in batch], dim=0),
        'cv_attention_mask': torch.cat([b['cv_attention_mask'] for b in batch], dim=0),
        'vac_input_ids': torch.cat([b['vac_input_ids'] for b in batch], dim=0),
        'vac_attention_mask': torch.cat([b['vac_attention_mask'] for b in batch], dim=0),
        'labels': torch.cat([b['labels'] for b in batch], dim=0),
        
        # Flatten the nested lists of strings
        'cvid': [c for b in batch for c in b['cvid']],
        'vacancy_id': [v for b in batch for v in b['vacancy_id']]
    }

In [5]:
trainloader = torch.load(f'../dataloaders/sentence_trainloader.pth',
                         weights_only=False)
valloader = torch.load(f'../dataloaders/sentence_valloader.pth',
                         weights_only=False)
testloader = torch.load(f'../dataloaders/sentence_testloader.pth',
                         weights_only=False)

In [6]:
import torch.nn.functional as F

def multiple_negatives_ranking_loss(cv_embeddings, vac_embeddings, temperature=0.05):
    """
    Computes contrastive loss. The diagonal contains the positive 'next sentences'.
    Everything else is an in-batch negative.
    """
    # Normalize embeddings so dot product equals cosine similarity
    cv_embeddings = F.normalize(cv_embeddings, p=2, dim=1)
    vac_embeddings = F.normalize(vac_embeddings, p=2, dim=1)
    
    # Compute similarity matrix: [batch_size, batch_size]
    scores = torch.matmul(cv_embeddings, vac_embeddings.T) / temperature
    
    # The true "next sentence" for cv[i] is vac[i], which is on the diagonal
    labels = torch.arange(scores.size(0), device=scores.device)
    
    # Standard Cross Entropy forces the diagonal toward 1 and off-diagonals toward 0
    return F.cross_entropy(scores, labels)

In [7]:
class text_ranker(torch.nn.Module):
    def __init__(self, pooling="mean"):
        super().__init__()
        self.model = AutoModel.from_pretrained("jjzha/dajobbert-base-uncased")
        self.pooling = pooling
        
        # Freeze the bottom 8 layers for speed
        for name, param in self.model.named_parameters():
            if 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < 8:  
                    param.requires_grad = False

    def pool_embeddings(self, outputs, attention_mask):
        if self.pooling == "mean":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            sum_embeddings = torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
            sum_mask = torch.clamp(input_mask_expanded.sum(1), min=1e-9) 
            return sum_embeddings / sum_mask
            
        elif self.pooling == "sum":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size())
            return torch.sum(outputs.last_hidden_state * input_mask_expanded, 1)
        
        elif self.pooling == "max":
            input_mask_expanded = attention_mask.unsqueeze(-1).expand(outputs.last_hidden_state.size()).bool()
            masked_embeddings = outputs.last_hidden_state * input_mask_expanded 
            embeddings, _ = torch.max(masked_embeddings, dim=1) 
            return embeddings

    def forward(self, batch_cv, batch_vac):
        cv_outputs = self.model(batch_cv['input_ids'], attention_mask=batch_cv['attention_mask'])
        cv_emb = self.pool_embeddings(cv_outputs, batch_cv['attention_mask'])
        
        vac_outputs = self.model(batch_vac['input_ids'], attention_mask=batch_vac['attention_mask'])
        vac_emb = self.pool_embeddings(vac_outputs, batch_vac['attention_mask'])
        
        # Return the raw representations
        return cv_emb, vac_emb

In [25]:
def train_loop(model, optimizer, trainloader):
    model.train()
    scaler = torch.cuda.amp.GradScaler() 
    epoch_losses = []
        
    for i, batch in enumerate(trainloader):
        batch_cv = {
            'input_ids': batch['cv_input_ids'].to(device),
            'attention_mask': batch['cv_attention_mask'].to(device)
        }
        batch_vac = {
            'input_ids': batch['vac_input_ids'].to(device),
            'attention_mask': batch['vac_attention_mask'].to(device)
        }

        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            # Get the representations
            cv_emb, vac_emb = model(batch_cv, batch_vac)
            
            # Compute MNRL (InfoNCE)
            loss = multiple_negatives_ranking_loss(cv_emb, vac_emb)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        
        scaler.step(optimizer)
        scaler.update()

        epoch_losses.append(loss.item())
        print(f"Batch: {i + 1}/{len(trainloader)}, Loss: {loss.item():.4f}    ", end="\r", flush=True)
        
    return epoch_losses


def val_loop(model, valloader, gender_map, area_map):
    model.eval() # Switch to evaluation mode
    
    # Track overall NDCG
    all_scores = []
    
    # Track NDCG separately by gender (Performance Disparity)
    ndcg_scores = {'Male': [], 'Female': [], 'Other': [], 'Unknown': []}
    
    # Track global counts for dataset-relative Disparate Visibility (Delta V)
    total_rural_dataset = 0
    total_items_dataset = 0
    total_rural_recommended = 0
    total_items_recommended = 0
    
    vacancy_pool_sizes = []
    
    with torch.no_grad():
        for i, batch in enumerate(valloader):
            print(f"Batch: {i + 1}/{len(valloader)}              ", end="\r")
            
            batch_cv = {
                'input_ids': batch['cv_input_ids'].to(device),
                'attention_mask': batch['cv_attention_mask'].to(device)
            }
            batch_vac = {
                'input_ids': batch['vac_input_ids'].to(device),
                'attention_mask': batch['vac_attention_mask'].to(device)
            }
            ground_truth = batch['labels'].to(device)
            
            # 1. Get the raw embeddings from the Bi-Encoder
            with torch.cuda.amp.autocast():
                cv_emb, vac_emb = model(batch_cv, batch_vac)
                
                # 2. Calculate the cosine similarity for the pairs
                y_pred_val = F.cosine_similarity(cv_emb, vac_emb)
                
            # Safely extract vacancy IDs before potential truncation
            vac_ids = batch.get('vacancy_id', [])
                
            if len(y_pred_val) > len(ground_truth):
                y_pred_val = y_pred_val[:len(ground_truth)]
                if len(vac_ids) > len(ground_truth):
                    vac_ids = vac_ids[:len(ground_truth)]
        
            # 3. Evaluate using the exact same NDCG logic
            y_true = ground_truth.detach().cpu().unsqueeze(0)
            y_score = y_pred_val.squeeze().unsqueeze(0).detach().cpu()
            
            batch_ndcg = ndcg_score(y_true, y_score, k=10)
            all_scores.append(batch_ndcg)
            
            # --- Gender Fairness (Utility) ---
            cvid_raw = batch.get("cvid", [None])
            candidate_id = cvid_raw[0][0] if isinstance(cvid_raw[0], (list, tuple)) else cvid_raw[0]
            
            gender = gender_map.get(candidate_id, 'Unknown')
            if candidate_id is not None:
                ndcg_scores[gender].append(batch_ndcg)
            
            # --- Geographic Fairness (Dataset-Relative Disparate Visibility Delta V) ---
            vacancy_pool_sizes.append(len(vac_ids))
            
            if len(vac_ids) > 0:
                batch_rural_count = sum(1 for v in vac_ids if area_map.get(int(float(v))) == 'Rural')
                total_rural_dataset += batch_rural_count
                total_items_dataset += len(vac_ids)
                
                actual_k = min(10, len(vac_ids))
                if actual_k > 0:
                    y_pred_squeeze = y_pred_val.squeeze()
                    if y_pred_squeeze.ndim == 0:
                        y_pred_squeeze = y_pred_squeeze.unsqueeze(0)
                        
                    top_k_indices = torch.topk(y_pred_squeeze, actual_k).indices.tolist()
                    if not isinstance(top_k_indices, list):
                        top_k_indices = [top_k_indices]
                        
                    top_k_vacs = [vac_ids[idx] for idx in top_k_indices]
                    top_k_rural_count = sum(1 for v in top_k_vacs if area_map.get(int(float(v))) == 'Rural')
                    
                    total_rural_recommended += top_k_rural_count
                    total_items_recommended += actual_k

    mean_male_ndcg = np.mean(ndcg_scores['Male']) if ndcg_scores['Male'] else 0.0
    mean_female_ndcg = np.mean(ndcg_scores['Female']) if ndcg_scores['Female'] else 0.0
    ndcg_gap = (mean_female_ndcg - mean_male_ndcg) if (len(ndcg_scores['Male']) > 0 and len(ndcg_scores['Female']) > 0) else None

    if total_items_dataset > 0 and total_items_recommended > 0:
        frac_dataset = total_rural_dataset / total_items_dataset
        frac_recommended = total_rural_recommended / total_items_recommended
        mean_disp_vis = frac_recommended - frac_dataset
    else:
        mean_disp_vis = None
            
    return all_scores, ndcg_gap, mean_disp_vis

In [26]:
def train_model(model, optimizer, scheduler, trainloader, valloader, gender_map, area_map, epochs=5, step_size=5):
    best_score = 0
       
    for epoch in range(epochs + 1):
        print(f"\nEpoch: {epoch}/{epochs}")

        # Train the model for the current epoch (Returns MNRL Loss)
        epoch_losses = train_loop(model, optimizer, trainloader)

        print(f"\nTraining Loss (MNRL): {np.mean(epoch_losses):.4f}")
        scheduler.step()

        # Evaluate the model (Returns NDCG and Fairness Metrics)
        val_ndcg_scores, final_ndcg_gap, final_disp_vis = val_loop(model, valloader, gender_map, area_map)
        
        current_ndcg = np.mean(val_ndcg_scores)
        print(f"\nTesting nDCG: {current_ndcg:.4f}")
        
        if final_ndcg_gap is None:
             print("Performance Disparity: N/A")
        else:
             print(f"Performance Disparity (Male - Female): {final_ndcg_gap:.4f}")
             
        if final_disp_vis is None:
             print("Disparate Visibility (Delta V): N/A")
        else:
             print(f"Disparate Visibility (Delta V): {final_disp_vis:.4f}")

        if current_ndcg > best_score:
            best_score = current_ndcg
            
    return best_score, final_ndcg_gap, final_disp_vis

In [27]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
torch.cuda.empty_cache() 
gc.collect()

df_gender = pd.read_csv("anonid_gender_mapping.csv")
gender_map = dict(zip(df_gender['anon_id'], df_gender['gender']))

df_location = pd.read_csv("job_area_mapping.csv")
area_map = dict(zip(df_location["humanjobid"], df_location["area"]))

best_config = {'learning_rate': 8.493296693542725e-05, 'pooling_method': 'mean', 'epochs': 5}

warnings.filterwarnings('ignore')

model = text_ranker(pooling=best_config["pooling_method"]).to(device)      
model.train()
    
optimizer = torch.optim.Adam(model.parameters(), lr=best_config["learning_rate"])
scheduler = StepLR(optimizer, step_size=3, gamma=0.1)

train_model(model, optimizer, scheduler, trainloader, testloader, gender_map, area_map,
            epochs=best_config["epochs"], step_size=5)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jjzha/dajobbert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
pooler.dense.bias                          | MISSING    | 
pooler.dense.weight                        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



Epoch: 0/5
Batch: 276/276, Loss: 0.6895    
Training Loss (MNRL): 0.9228
Batch: 37/37              
Testing nDCG: 0.3867
Performance Disparity (Male - Female): -0.1248
Disparate Visibility (Delta V): 0.0114

Epoch: 1/5
Batch: 276/276, Loss: 1.1025    
Training Loss (MNRL): 0.9216
Batch: 37/37              
Testing nDCG: 0.3512
Performance Disparity (Male - Female): 0.0195
Disparate Visibility (Delta V): 0.0005

Epoch: 2/5
Batch: 276/276, Loss: 1.3887    
Training Loss (MNRL): 0.9216
Batch: 37/37              
Testing nDCG: 0.3378
Performance Disparity (Male - Female): -0.0449
Disparate Visibility (Delta V): 0.0005

Epoch: 3/5
Batch: 276/276, Loss: 2.4847    
Training Loss (MNRL): 0.9215
Batch: 37/37              
Testing nDCG: 0.3386
Performance Disparity (Male - Female): -0.0399
Disparate Visibility (Delta V): 0.0060

Epoch: 4/5
Batch: 276/276, Loss: 0.0000    
Training Loss (MNRL): 0.9213
Batch: 37/37              
Testing nDCG: 0.3328
Performance Disparity (Male - Female): -0.0468


(0.3867379302866686, -0.056051490594197784, 0.014058244290802435)